In [5]:
import log_util
import requests, json, time
from __init__ import version, log_to_directory
from typing import Any, Dict, List, Optional

try: 
    from pathlib import Path
except:
    from pathlib2 import Path


logger = log_util.get_logger(__name__)
log_to_directory("logs")

In [6]:
class ElsClient:
    #TODO: Pasar todo esto a un .env
    __url_base = 'https://api.elsevier.com/'
    __user_agent = "scopus-metrics-utb-v%s" % version
    __min_req_interval = 1
    __ts_last_req = time.time()

    def __init__(
        self, 
        api_key: str, 
        inst_token: Optional[str] = None,
        num_res: Optional[int] = 25, 
        local_dir: Optional[str | Path] = None
    ):
        self.api_key = api_key
        self.inst_token = inst_token
        self.num_res = num_res

        if local_dir is None: 
            self.local_dir = Path.cwd()  / 'data'
        else:
            self.local_dir = Path(local_dir)

        if not self.local_dir.exists():
            self.local_dir.mkdir()


    # properties
    @property
    def api_key(self):
        """Get the apiKey for the client instance"""
        return self._api_key
    
    @api_key.setter
    def api_key(self, api_key):
        """Set the apiKey for the client instance"""
        self._api_key = api_key

    @property
    def inst_token(self):
        """Get the instToken for the clien instance"""
        return self._inst_token
    
    @inst_token.setter
    def inst_token(self, inst_token):
        """Set the instToken for the client instance"""
        self._inst_token = inst_token

    @property
    def num_res(self):
        """Gets the max. number of results to be used by the client instance"""
        return self._num_res
    
    @num_res.setter
    def num_res(self, numRes):
        """Sets the max. number of results to be used by the client instance"""
        self._num_res = numRes

    @property
    def local_dir(self):
        """Gets the currently configured local path to write data to."""
        return self._local_dir

    @local_dir.setter
    def local_dir(self, path):
        """Sets the local path to write data to."""
        self._local_dir = Path(path)

    @property
    def req_status(self):
        '''Return the status of the request response, '''
        return {'status_code': self._status_code, 'status_msg': self._status_msg}
    
    # acces functions
    def getBaseURL(self):
        """Returns the ELSAPI base URL currently configured for the client"""
        return self.__url_base

    # request/response functions
    def exec_request(self, URL, params: Optional[Dict[str, Any]] = None):
        """Sends the actual request; returns response"""

        interval = time.time() - self.__ts_last_req
        if (interval < self.__min_req_interval):
            time.sleep(self.__min_req_interval - interval)

        # Construct and execute request
        headers = {
            "X-ELS-APIKey"  : self.api_key,
            "User-Agent"    : self.__user_agent,
            "Accept"        : 'application/json'
        }

        if self.inst_token:
            headers["X-ELS-Insttoken"] = self.inst_token
        logger.info('Sending GET request to ' + URL)

        r = requests.get(
            URL, 
            headers=headers, 
            params=params
        )

        self.__ts_last_req = time.time()
        self._status_code = r.status_code

        if r.status_code == 200:
            self._status_msg = 'data retrieved'
            print(self._status_msg)
        else:
            msg = (
                f"HTTP {r.status_code} Error from {URL} "
                f"and using headers {headers}: {r.text}"
            )

            self._status_msg = msg
            logger.error(msg)           
            raise requests.HTTPError(msg)



In [7]:
from dotenv import load_dotenv
import os


load_dotenv()
api = os.getenv("ELSEVIER_APIKEY")
token = os.getenv("ELSEVIER_INSTTOKEN")

intento = ElsClient(
    api_key= api, 
    inst_token= token
)

In [8]:
intento.exec_request(intento.getBaseURL())

data retrieved


## __Prueba de API__

Ahora vamos a intentar hacer una consulta a la API de Scopus para obtener información en especifico sobre los journals. 

In [9]:
__url_base = 'https://api.elsevier.com/'
__user_agent = "scopus-metrics-utb-v%s" % version

url_final = 'https://api.elsevier.com/content/serial/title'

headers = {
    "X-ELS-APIKey"  : api,
    "X-ELS-Insttoken": token,
    "User-Agent"    : __user_agent,
    "Accept"        : 'application/json',
}

r =requests.get(
    url_final,
    headers=headers
)

print(r.status_code)
print(r.text)


200
{"serial-metadata-response":{"link": [{"@_fa": "true", "@ref": "self", "@href": "https://api.elsevier.com/content/serial/title?start=0&count=25&apiKey=0a94086b522c7bea8b3f14343d4726fe&view=STANDARD", "@type": "application/json"},{"@_fa": "true", "@ref": "first", "@href": "https://api.elsevier.com/content/serial/title?start=0&count=25&apiKey=0a94086b522c7bea8b3f14343d4726fe&view=STANDARD", "@type": "application/json"},{"@_fa": "true", "@ref": "next", "@href": "https://api.elsevier.com/content/serial/title?start=25&count=25&apiKey=0a94086b522c7bea8b3f14343d4726fe&view=STANDARD", "@type": "application/json"},{"@_fa": "true", "@ref": "last", "@href": "https://api.elsevier.com/content/serial/title?start=9975&count=25&apiKey=0a94086b522c7bea8b3f14343d4726fe&view=STANDARD", "@type": "application/json"}],"entry": [{"@_fa": "true", "dc:title":"1700-tal: Nordic Journal for Eighteenth-Century Studies","dc:publisher":"Swedish Society for Eighteenth-Century Studies","coverageStartYear":"2016","

In [17]:
with open('prueba.json', 'w', encoding='utf-8') as f:
    json.dump(r.json(), f, ensure_ascii=False, indent=4)